# Midweek Analysis
Testing whether rest days / free midweeks are an advantage in Premier League matches.

In [19]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf

DB_PATH = '../../infra/data/db/fotmob.db'
FIXTURES_PATH = '../../infra/data/json/fbref/fixtures_manual.csv'

## 1. Load All Fixtures (all competitions — used to compute rest days)

In [20]:
fixtures = pd.read_csv(FIXTURES_PATH, parse_dates=['date'])
fixtures = fixtures.dropna(subset=['home_goals', 'away_goals']).sort_values('date').reset_index(drop=True)

print(f"All completed fixtures: {len(fixtures)}")
print(fixtures['league_name'].value_counts())

All completed fixtures: 8626
league_name
Premier League       3359
Europa League        1592
FA Cup               1184
Champions League     1118
Conference League     714
EFL Cup               659
Name: count, dtype: int64


## 2. Load ClubElo Ratings

In [21]:
conn = sqlite3.connect(DB_PATH)
elo_raw = pd.read_sql(
    "SELECT Club, Elo, [From] as elo_from FROM clubelo_ratings WHERE Country = 'ENG'",
    conn
)
conn.close()

elo_raw['elo_from'] = pd.to_datetime(elo_raw['elo_from'])
elo = (
    elo_raw
    .drop_duplicates(subset=['Club', 'elo_from'])
    .sort_values(['Club', 'elo_from'])
    .reset_index(drop=True)
)
print(f"ClubElo rows (deduped): {len(elo)}")

ClubElo rows (deduped): 14521


## 3. Team Name Mapping (FBRef -> ClubElo)

In [22]:
base_map = pd.read_csv('../../infra/data/collectors/clubelo/clubelo_fbref_team_name_mapping.csv')
fbref_to_clubelo = dict(zip(base_map['fbref_name'], base_map['clubelo_name']))

fbref_to_clubelo.update({
    'Huddersfield Town': 'Huddersfield',
    'Hull City':         'Hull',
    'Ipswich Town':      'Ipswich',
    'Leicester City':    'Leicester',
    'Luton Town':        'Luton',
    'Manchester City':   'Man City',
    'Manchester Utd':    'Man United',
    'Newcastle United':  'Newcastle',
    "Nott'ham Forest":   'Forest',
    'Nottingham Forest': 'Forest',
    'Norwich City':      'Norwich',
    'Sheffield United':  'Sheffield United',
    'Stoke City':        'Stoke',
    'Swansea City':      'Swansea',
    'Tottenham Hotspur': 'Tottenham',
    'West Ham United':   'West Ham',
    'Cardiff City':      'Cardiff',
})

## 4. Calculate Days Rest (across all competitions)

In [23]:
# Reshape to one row per team per match (all competitions)
home_rows = fixtures[['date', 'season', 'home_team']].rename(columns={'home_team': 'team'})
away_rows = fixtures[['date', 'season', 'away_team']].rename(columns={'away_team': 'team'})
all_team_matches = pd.concat([home_rows, away_rows], ignore_index=True)
all_team_matches = all_team_matches.sort_values(['team', 'date']).reset_index(drop=True)

# Days since last match (any competition) — NaN for first-ever match
all_team_matches['prev_match_date'] = all_team_matches.groupby('team')['date'].shift(1)
all_team_matches['days_rest'] = (all_team_matches['date'] - all_team_matches['prev_match_date']).dt.days

rest_lookup = all_team_matches.set_index(['team', 'date'])['days_rest']

print(f"Rest day stats (all teams, all comps):")
print(all_team_matches['days_rest'].describe())

Rest day stats (all teams, all comps):
count    16843.000000
mean        40.614795
std        119.560774
min          2.000000
25%          5.000000
50%          8.000000
75%         21.000000
max       2919.000000
Name: days_rest, dtype: float64


## 5. Build PL Team-Match Dataset

In [24]:
def lookup_elo_asof(df: pd.DataFrame, team_col: str, date_col: str, elo_df: pd.DataFrame) -> pd.Series:
    """Last-known ClubElo rating on or before the match date."""
    results = []
    for club, group in df.groupby(team_col, sort=False):
        club_elo = elo_df[elo_df['Club'] == club][['elo_from', 'Elo']].sort_values('elo_from')
        if club_elo.empty:
            results.append(pd.Series(np.nan, index=group.index, name='_elo'))
            continue
        matched = pd.merge_asof(
            group[[date_col]].sort_values(date_col),
            club_elo,
            left_on=date_col,
            right_on='elo_from',
            direction='backward'
        )
        matched.index = group.sort_values(date_col).index
        results.append(matched['Elo'].rename('_elo'))
    return pd.concat(results).reindex(df.index)


pl = fixtures[fixtures['league_name'] == 'Premier League'].copy().reset_index(drop=True)

# Reshape to one row per team per PL match
home = pl.assign(
    team=pl['home_team'], opponent=pl['away_team'],
    goals_for=pl['home_goals'], goals_against=pl['away_goals'],
    is_home=1
)
away = pl.assign(
    team=pl['away_team'], opponent=pl['home_team'],
    goals_for=pl['away_goals'], goals_against=pl['home_goals'],
    is_home=0
)
tm = pd.concat([home, away], ignore_index=True)
tm['goal_diff'] = tm['goals_for'] - tm['goals_against']

# Map elo names
tm['team_elo_name']     = tm['team'].map(fbref_to_clubelo)
tm['opponent_elo_name'] = tm['opponent'].map(fbref_to_clubelo)

# Join elo ratings
tm['team_elo']     = lookup_elo_asof(tm, 'team_elo_name',     'date', elo)
tm['opponent_elo'] = lookup_elo_asof(tm, 'opponent_elo_name', 'date', elo)

# Join rest days
tm['days_rest']          = tm.set_index(['team', 'date']).index.map(rest_lookup)
tm['opponent_days_rest'] = tm.set_index(['opponent', 'date']).index.map(rest_lookup)

# Drop first PL match of each season per team (no prior-season rest baseline)
tm = tm.sort_values(['team', 'date'])
tm['season_match_num'] = tm.groupby(['team', 'season']).cumcount()
tm = tm[tm['season_match_num'] > 0].reset_index(drop=True)

print(f"Team-match rows: {len(tm)}")
print(f"days_rest missing: {tm['days_rest'].isna().sum()} | opp missing: {tm['opponent_days_rest'].isna().sum()}")
tm[['date', 'season', 'team', 'opponent', 'is_home', 'goal_diff', 'team_elo', 'opponent_elo', 'days_rest', 'opponent_days_rest']].head(10)

Team-match rows: 6538
days_rest missing: 0 | opp missing: 0


,date,season,team,opponent,is_home,goal_diff,team_elo,opponent_elo,days_rest,opponent_days_rest
0,2017-08-19,2017-2018,Arsenal,Stoke City,0,-1,1851.603760,1658.446777,8.0,7.0
1,2017-08-27,2017-2018,Arsenal,Liverpool,0,-4,1844.331787,1846.845825,8.0,8.0
2,2017-09-09,2017-2018,Arsenal,Bournemouth,1,3,1834.606934,1633.596313,13.0,14.0
3,2017-09-17,2017-2018,Arsenal,Chelsea,0,0,1838.519653,1918.657715,3.0,5.0
4,2017-09-25,2017-2018,Arsenal,West Brom,1,2,1848.001099,1653.421021,5.0,5.0
5,2017-10-01,2017-2018,Arsenal,Brighton,1,2,1851.626343,1599.735352,3.0,7.0
6,2017-10-14,2017-2018,Arsenal,Watford,0,-1,1865.877319,1657.044434,13.0,14.0
7,2017-10-22,2017-2018,Arsenal,Everton,0,3,1854.291626,1729.119873,3.0,3.0
8,2017-10-28,2017-2018,Arsenal,Swansea City,1,1,1871.817627,1652.681152,4.0,4.0
9,2017-11-05,2017-2018,Arsenal,Manchester City,0,-2,1874.000244,1952.713745,3.0,4.0


## 6. Baseline OLS Model

In [25]:
model_df = tm[[
    'goal_diff', 'is_home', 'team_elo', 'opponent_elo', 'days_rest', 'opponent_days_rest'
]].dropna().copy()

print(f"Fitting on {len(model_df)} rows")

formula = 'goal_diff ~ is_home + team_elo + opponent_elo + days_rest + opponent_days_rest'
result = smf.ols(formula, data=model_df).fit()
print(result.summary())

Fitting on 6538 rows
                            OLS Regression Results                            
Dep. Variable:              goal_diff   R-squared:                       0.222
Model:                            OLS   Adj. R-squared:                  0.221
Method:                 Least Squares   F-statistic:                     371.8
Date:                Sat, 18 Apr 2026   Prob (F-statistic):               0.00
Time:                        05:08:34   Log-Likelihood:                -12730.
No. Observations:                6538   AIC:                         2.547e+04
Df Residuals:                    6532   BIC:                         2.551e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept      

## 7. Rest Day Distribution Check

In [26]:
print("Team days_rest distribution:")
print(tm['days_rest'].value_counts().sort_index().head(20))
print()
print("Mean goal_diff by days_rest bucket:")
tm['rest_bucket'] = pd.cut(tm['days_rest'], bins=[0, 4, 6, 8, 14, 999], labels=['<=4', '5-6', '7-8', '9-14', '15+'])
print(tm.groupby('rest_bucket', observed=True)['goal_diff'].agg(['mean', 'count']))

Team days_rest distribution:
days_rest
2.0       51
3.0     1480
4.0     1051
5.0      592
6.0      612
7.0     1308
8.0      491
9.0      106
10.0      36
11.0      44
12.0      14
13.0     174
14.0     319
15.0     103
16.0      50
17.0      23
18.0      16
19.0       4
20.0       2
21.0      19
Name: count, dtype: int64

Mean goal_diff by days_rest bucket:
                 mean  count
rest_bucket                 
<=4          0.142912   2582
5-6          0.005814   1204
7-8         -0.170094   1799
9-14        -0.054834    693
15+         -0.119231    260


## 8. Bucketed Rest Model
Short rest = ≤3 days, Mid rest = 4–7 days (reference), Long rest = 8+ days.

In [27]:
def rest_bucket(days):
    if pd.isna(days):
        return np.nan
    elif days <= 3:
        return 'short'
    elif days <= 7:
        return 'mid'
    else:
        return 'long'

tm['rest_cat']     = pd.Categorical(tm['days_rest'].map(rest_bucket),          categories=['mid', 'short', 'long'])
tm['opp_rest_cat'] = pd.Categorical(tm['opponent_days_rest'].map(rest_bucket), categories=['mid', 'short', 'long'])

print('Team rest category counts:')
print(tm['rest_cat'].value_counts())
print()
print('Mean goal_diff by rest_cat:')
print(tm.groupby('rest_cat', observed=True)['goal_diff'].agg(['mean', 'count']))

Team rest category counts:
rest_cat
mid      3563
short    1531
long     1444
Name: count, dtype: int64

Mean goal_diff by rest_cat:
              mean  count
rest_cat                 
mid      -0.058378   3563
short     0.281515   1531
long     -0.153740   1444


In [28]:
model_df2 = tm[[
    'goal_diff', 'is_home', 'team_elo', 'opponent_elo', 'rest_cat', 'opp_rest_cat'
]].dropna().copy()

print(f'Fitting on {len(model_df2)} rows')

formula2 = ('goal_diff ~ is_home + team_elo + opponent_elo'
            ' + C(rest_cat, Treatment(reference="mid"))'
            ' + C(opp_rest_cat, Treatment(reference="mid"))')
result2 = smf.ols(formula2, data=model_df2).fit()
print(result2.summary())

Fitting on 6538 rows
                            OLS Regression Results                            
Dep. Variable:              goal_diff   R-squared:                       0.222
Model:                            OLS   Adj. R-squared:                  0.221
Method:                 Least Squares   F-statistic:                     266.7
Date:                Sat, 18 Apr 2026   Prob (F-statistic):               0.00
Time:                        05:08:34   Log-Likelihood:                -12727.
No. Observations:                6538   AIC:                         2.547e+04
Df Residuals:                    6530   BIC:                         2.552e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                                           coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------

## 9. Interaction Model
Team rest × opponent rest — reference cell is mid vs mid.

In [29]:
# Cross-tab of mean goal_diff across rest matchups
print('Mean goal_diff by rest matchup (team rest vs opp rest):')
print(
    tm.groupby(['rest_cat', 'opp_rest_cat'], observed=True)['goal_diff']
    .agg(['mean', 'count'])
    .round(3)
)


Mean goal_diff by rest matchup (team rest vs opp rest):
                        mean  count
rest_cat opp_rest_cat              
mid      mid          -0.000   2513
         short        -0.442    721
         long          0.340    329
short    mid           0.442    721
         short         0.000    614
         long          0.571    196
long     mid          -0.337    326
         short        -0.571    196
         long          0.000    922


In [30]:
model_df3 = tm[[
    'goal_diff', 'is_home', 'team_elo', 'opponent_elo', 'rest_cat', 'opp_rest_cat'
]].dropna().copy()

print(f'Fitting on {len(model_df3)} rows')

formula3 = (
    'goal_diff ~ is_home + team_elo + opponent_elo'
    ' + C(rest_cat, Treatment(reference="mid")) * C(opp_rest_cat, Treatment(reference="mid"))'
)
result3 = smf.ols(formula3, data=model_df3).fit()
print(result3.summary())


Fitting on 6538 rows
                            OLS Regression Results                            
Dep. Variable:              goal_diff   R-squared:                       0.223
Model:                            OLS   Adj. R-squared:                  0.221
Method:                 Least Squares   F-statistic:                     169.8
Date:                Sat, 18 Apr 2026   Prob (F-statistic):               0.00
Time:                        05:08:34   Log-Likelihood:                -12726.
No. Observations:                6538   AIC:                         2.548e+04
Df Residuals:                    6526   BIC:                         2.556e+04
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                                                                                            coef    std err          t      P>|t|      [0.025      0.975]
-------------------

In [31]:
# Cleaner view: predicted goal_diff for each rest matchup (holding elo diff=0, neutral venue)
from itertools import product

combos = pd.DataFrame(
    list(product(['mid', 'short', 'long'], ['mid', 'short', 'long'])),
    columns=['rest_cat', 'opp_rest_cat']
)
combos['rest_cat']     = pd.Categorical(combos['rest_cat'],     categories=['mid','short','long'])
combos['opp_rest_cat'] = pd.Categorical(combos['opp_rest_cat'], categories=['mid','short','long'])
combos['is_home']      = 0
combos['team_elo']     = tm['team_elo'].mean()
combos['opponent_elo'] = tm['team_elo'].mean()

combos['predicted_goal_diff'] = result3.predict(combos)

pivot = combos.pivot(index='rest_cat', columns='opp_rest_cat', values='predicted_goal_diff').round(3)
print('Predicted goal diff (rows=team rest, cols=opp rest, neutral/equal elo):')
print(pivot)


Predicted goal diff (rows=team rest, cols=opp rest, neutral/equal elo):
opp_rest_cat    mid  short   long
rest_cat                         
mid          -0.259 -0.380 -0.178
short        -0.137 -0.259 -0.230
long         -0.330 -0.288 -0.259
